# SuperRaman classifier training

**SERS-D2DNet + SuperRaman · v0.3.1**

[Notebook guide](../docs/NOTEBOOK_GUIDE.md) · [Paper alignment](../docs/PAPER_IMPLEMENTATION_MAP.md) · [README](../README.md)

## Goal
Control the reusable Renishaw model independently of translator fitting.

## Setup
Public notebooks start without saved execution outputs. Select the Python environment where `.[notebooks]` is installed. Edit `CONFIG` below to select the demo or your research configuration. Every notebook opens the same persistent run. Configuration changes require a new `run_directory`. The default executes a synthetic, one-epoch, width-2 demonstration across all four portable devices.

In [ ]:
CONFIG = "configs/workflow_demo.json"
RUN_RESEARCH_TRAINING = False

In [ ]:
import numpy as np
from sersd2d_superraman import __version__
from sersd2d_superraman.provenance import read_json
from sersd2d_superraman.data import load_batch
from sersd2d_superraman.notebook import (
    open_workflow,
    display_table,
    setup_plotting,
    training_plot,
    spectrum_plot,
    confusion_plot,
)

workflow = open_workflow(CONFIG)
PROJECT, settings = workflow.project, workflow.config
MODE = settings["mode"]
RUN_TRAINING = MODE == "demo" or RUN_RESEARCH_TRAINING
setup_plotting()
display_table(
    ["Control", "Value"],
    [
        ["Software", __version__],
        ["Mode", MODE],
        ["Devices", ", ".join(settings["sources"])],
        [
            "Translation width / epochs",
            f"{settings['translation_config']['model']['width']} / {settings['translation_config']['training']['epochs']}",
        ],
        ["Classifier epochs", settings["classifier_config"]["training"]["epochs"]],
        [
            "Classifier / translation device",
            f"{settings['classifier_device']} / {settings['translation_device']}",
        ],
        ["Research training requested", RUN_RESEARCH_TRAINING],
    ],
)

### Read the execution scope
The demo verifies software behavior. Its plots and scores are synthetic and have no scientific performance interpretation. Research mode uses authorized preprocessed arrays and the recovered width-128 graph. Published-result reproduction remains unresolved; review the paper alignment document before reporting results.

## Dataset access
The [Multi-Instrument SERS Dataset](https://zenodo.org/records/23372983) is hosted on Zenodo with restricted file access. Request access with a brief intended-use statement. After approval, download/extract the files and follow the [local data setup guide](../docs/DATA.md). The synthetic demo requires no dataset access. In a terminal, `sers-pipeline dataset-info` shows the access and configuration steps.


## Steps
### 1. Prepare and audit the input roles
The interface accepts preprocessed `[N,1400]` spectra in `[0,1]`. Rows retain device, analyte, sample identity, optional Raman axis, and optional acquisition group. Class-matched training pairs are physically unpaired. Split checks keep duplicates and acquisition groups together.

In [ ]:
counts = workflow.prepare()
display_table(
    ["Device", "Fit", "Validation", "Test"],
    [[d, r["train"], r["validation"], r["test"]] for d, r in counts.items()],
)

### 3. Train the reusable Renishaw classifier
SuperRaman learns from the Renishaw fit role and selects checkpoints on independent Renishaw validation. The maintained implementation uses cross-entropy on logits and legacy fractional shifts. These differ from the paper's MAE/integer-shift wording and are identified in every configuration and the alignment map.

In [ ]:
classifier_run = workflow.train_classifier() if RUN_TRAINING else None
if not RUN_TRAINING:
    print("Research classifier training is disabled. Set RUN_RESEARCH_TRAINING = True in Setup to run it.")
if classifier_run:
    print(
        {key: classifier_run[key] for key in ("best_epoch", "best_validation_loss", "objective", "synthetic")}
    )

### Inspect the trained classifier contract

In [ ]:
checkpoint = workflow.output / "classification/checkpoint/metadata.json"
if checkpoint.is_file():
    metadata = read_json(checkpoint)
    print(
        {
            k: metadata[k]
            for k in ("parameters", "architecture_id", "input_transform", "paper_reproduction_certified")
        }
    )

## Checks
Artifact reuse checks package-source and file hashes plus input/configuration identities. Full worker logs are stored under the run directory in logs/. Test rows are evaluated after checkpoint selection; they do not choose the models.

In [ ]:
display_table(["Stage", "Status"], workflow.status().items())

## Next Steps
For real experiments, copy `configs/workflow_research.json` to `configs/local/my_run.json`, set the data path and a new run directory, and select it in `CONFIG`. Set RUN_RESEARCH_TRAINING = True explicitly after reviewing the displayed settings. See the [notebook guide](../docs/NOTEBOOK_GUIDE.md) for stage navigation and interpretation.